# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an in-depth guide for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined and described using the Croissant schema, which is available at the URL provided below. All exploration in this notebook references elements (record sets, fields, columns) by their unique `@id` fields, as required by the Croissant specification.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Version: {getattr(meta, 'version', None)}")
print(f"Citation: {getattr(meta, 'citeAs', None)}")

## 2. Data Overview
Review available record sets, their fields, and corresponding `@id`s in the dataset metadata. All further exploration will reference entities by these `@id`s.

In [ ]:
# List all record sets with their @id and name
print('Record Sets:')
for rs in getattr(meta, 'record_set', []):
    print(f"@id: {rs.id} | name: {rs.name}")
    # List all fields in this record set
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for field in rs.fields:
            print(f"    @id: {field.id} | name: {field.name} | dataType: {getattr(field, 'data_type', None)}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Collect all record set IDs dynamically
record_sets = [rs.id for rs in getattr(meta, 'record_set', [])]
dataframes = dict()

# Preview records in each record set (first 2 records)
for record_set_id in record_sets:
    print(f"\nPreview of first two records in '{record_set_id}':")
    records_iter = dataset.records(record_set=record_set_id)
    for i, rec in enumerate(records_iter):
        if i >= 2:
            break
        print(rec)

# Load each record set into a DataFrame, if not too large
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nDataFrame columns for '{record_set_id}': {df.columns.tolist()}")
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)

This section demonstrates filtering, normalization, and grouping on an example record set (using its `@id`). All operations reference fields by their Croissant `@id`.

_For this example, we will search for a record set likely to contain protein abundance data with a numeric field (e.g., coverage, MW, or normalized abundance)._

In [ ]:
# Select the first record set with numeric fields for demonstration
selected_record_set, numeric_field_id, group_field_id = None, None, None
# Try to find a suitable record set and field
for rs in getattr(meta, 'record_set', []):
    for field in getattr(rs, 'fields', []):
        # simple heuristic: look for "coverage", "abundance", "MW", or a field defined as Float/Integer/Number
        dt = getattr(field, 'data_type', '').lower()
        field_name = getattr(field, 'name', '').lower()
        if dt in ['float', 'integer', 'number'] or any(kword in field_name for kword in ['coverage', 'mw', 'abundance']):
            selected_record_set = rs.id
            numeric_field_id = field.id
            # Try to pick a group-by field that's categorical
            for f2 in getattr(rs, 'fields', []):
                fname = getattr(f2, 'name', '').lower()
                ftype = getattr(f2, 'data_type', '').lower()
                if (ftype == 'text' or 'description' in fname) and f2.id != field.id:
                    group_field_id = f2.id
                    break
            break
    if selected_record_set:
        break

if not selected_record_set:
    raise RuntimeError('No numeric field found in any record set!')

# Show selected record set and fields
print(f"Selected record set for EDA: {selected_record_set}")
print(f"Numeric field (@id): {numeric_field_id}")
if group_field_id:
    print(f"Group field (@id): {group_field_id}")

# EDA Step: Filter by a threshold (e.g., >10)
if selected_record_set in dataframes:
    df = dataframes[selected_record_set]
    # Ensure the column is converted to numeric if possible (Croissant column names are @ids)
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where '{numeric_field_id}' > {threshold} (first 5):")
    print(filtered_df.head())

    # Normalize the numeric field
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - mean) / std
    print(f"\nNormalized '{numeric_field_id}' (first 5):")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Grouped aggregation if group_field is found
    if group_field_id and group_field_id in filtered_df:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nMean of '{numeric_field_id}' grouped by '{group_field_id}':")
        print(grouped.head())
else:
    print(f"DataFrame for record set '{selected_record_set}' not found.")

## 5. Visualization
Visualize the distribution of the selected numeric field. Additional visualizations (e.g., by group or for other record sets) can be added as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set in dataframes:
    df = dataframes[selected_record_set]
    col = numeric_field_id
    # Plot histogram
    plt.figure(figsize=(8, 5))
    sns.histplot(df[col].dropna(), kde=True, bins=30)
    plt.title(f"Histogram of {col} in record set {selected_record_set}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.show()

    # If group field exists, show aggregated bar plot
    if group_field_id and group_field_id in df:
        group_means = df.groupby(group_field_id)[col].mean().sort_values(ascending=False)[:10]
        plt.figure(figsize=(10, 6))
        sns.barplot(x=group_means.values, y=group_means.index, orient='h')
        plt.title(f"Top 10 mean {col} grouped by {group_field_id}")
        plt.xlabel(f"Mean {col}")
        plt.ylabel(group_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to explore and process the 
[FAIR² Mass Spectrometry](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the Croissant schema and `mlcroissant`.

- All dataset entities (record sets, fields, columns) were referenced by their `@id`, ensuring compatibility and reproducibility.
- We programmatically extracted data from all record sets, selected relevant fields for quantitative analysis, and visualized the data distribution.

_Feel free to extend this notebook by focusing on additional record sets, fields, and deeper domain-specific analyses relevant to the research questions at hand!_